In [2]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B")
msgs = [{"role":"user","content":"Hi"},{"role":"assistant","content":"Hello!"}]
text = tok.apply_chat_template(msgs, tokenize=False)
print(repr(text))

'<|im_start|>user\nHi<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nHello!<|im_end|>\n'


In [9]:
enc = tok(text, add_special_tokens=False, return_offsets_mapping=True)
for t , off in zip(enc["input_ids"], enc["offset_mapping"]):
    print(t, off, repr(text[off[0]:off[1]]), type(off))

151644 (0, 12) '<|im_start|>' <class 'tuple'>
872 (12, 16) 'user' <class 'tuple'>
198 (16, 17) '\n' <class 'tuple'>
13048 (17, 19) 'Hi' <class 'tuple'>
151645 (19, 29) '<|im_end|>' <class 'tuple'>
198 (29, 30) '\n' <class 'tuple'>
151644 (30, 42) '<|im_start|>' <class 'tuple'>
77091 (42, 51) 'assistant' <class 'tuple'>
198 (51, 52) '\n' <class 'tuple'>
151667 (52, 59) '<think>' <class 'tuple'>
271 (59, 61) '\n\n' <class 'tuple'>
151668 (61, 69) '</think>' <class 'tuple'>
271 (69, 71) '\n\n' <class 'tuple'>
9707 (71, 76) 'Hello' <class 'tuple'>
0 (76, 77) '!' <class 'tuple'>
151645 (77, 87) '<|im_end|>' <class 'tuple'>
198 (87, 88) '\n' <class 'tuple'>


In [3]:
from transformers import AutoTokenizer

msgs = [{"role": "user", "content": "Weather?"},
        {"role": "assistant", "reasoning_content": "User wants weather, call the tool.", "content": "Let me check."}]

for name in ["Qwen/Qwen3-4B", "unsloth/Llama-3.2-3B-Instruct"]:
    tok = AutoTokenizer.from_pretrained(name)
    print(f"====={name}\n{tok.apply_chat_template(msgs, tokenize=False)}")

/opt/conda/envs/vllm_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=====Qwen/Qwen3-4B
<|im_start|>user
Weather?<|im_end|>
<|im_start|>assistant
<think>
User wants weather, call the tool.
</think>

Let me check.<|im_end|>

=====unsloth/Llama-3.2-3B-Instruct
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 09 Oct 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

Weather?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Let me check.<|eot_id|>


### MODEL ADAPTERS TEST

In [4]:
from transformers import AutoTokenizer
from sft.data.chat_tokenizer import get_adapter

In [5]:
msgs = [{"role": "user", "content": "Weather?"},
        {"role": "assistant", "reasoning_content": "Call the tool.", "content": "Let me check.", "train": True}]
for name in ["Qwen/Qwen3-4B", "unsloth/Llama-3.2-3B-Instruct"]:
    a = get_adapter(AutoTokenizer.from_pretrained(name))
    out, tools = a.prepare(msgs, [])
    print(name, "|", a.name, a.reasoning, "|", out[1], "| tools:", tools, "| kwargs:", a.template_kwargs(out))
print("input unchanged:", msgs[1])

Qwen/Qwen3-4B | qwen3 native | {'role': 'assistant', 'content': 'Let me check.', 'reasoning_content': 'Call the tool.'} | tools: None | kwargs: {'enable_thinking': True}
unsloth/Llama-3.2-3B-Instruct | default inline | {'role': 'assistant', 'content': '<think>\nCall the tool.\n</think>\n\nLet me check.'} | tools: None | kwargs: {}
input unchanged: {'role': 'assistant', 'reasoning_content': 'Call the tool.', 'content': 'Let me check.', 'train': True}


In [1]:
from transformers import AutoTokenizer
from sft.data.chat_tokenizer import ChatTokenizer, _rstrip_end
ct = ChatTokenizer(AutoTokenizer.from_pretrained("Qwen/Qwen3-4B"))
tools = [{"type":"function","function":{"name":"get_weather","description":"Get weather.","parameters":{"type":"object","properties":{"city":{"type":"string"}},"required":["city"]}}}]
msgs = [{"role":"user","content":"Weather in Paris?"},
 {"role":"assistant","reasoning_content":"Call the tool.","content":"","tool_calls":[{"id":"c1","type":"function","function":{"name":"get_weather","arguments":{"city":"Paris"}}}]},
 {"role":"tool","tool_call_id":"c1","name":"get_weather","content":"18C, sunny"},
 {"role":"assistant","reasoning_content":"Got it.","content":"It's 18C and sunny in Paris."}]


/opt/conda/envs/vllm_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Qwen2Tokenizer(name_or_path='Qwen/Qwen3-4B', vocab_size=151643, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151649: AddedToken("<|box_end|>", rstrip=False, lstrip=F

In [3]:
m, t = ct.adapter.prepare(msgs, tools)
kw = ct.adapter.template_kwargs(m)

In [7]:
for i in (1, 3):
    p, u = ct._turn_span(m, t, kw, i)
    e = _rstrip_end(u, len(p), len(u))
    print(i, "prefix ok:", u.startswith(p), "| span:", repr(u[len(p):e]), "| stripped:", repr(u[e:]))

1 prefix ok: True | span: '<think>\nCall the tool.\n</think>\n\n<tool_call>\n{"name": "get_weather", "arguments": {"city": "Paris"}}\n</tool_call><|im_end|>' | stripped: '\n'
3 prefix ok: True | span: "<think>\nGot it.\n</think>\n\nIt's 18C and sunny in Paris.<|im_end|>" | stripped: '\n'


### CHAT_TOKENIZE.PY full test

In [1]:
import copy
from transformers import AutoTokenizer
from sft.data.chat_tokenizer import ChatTokenizer, TokenizerConfig
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B")
tools = [{"type":"function","function":{"name":"get_weather","description":"Get weather.","parameters":{"type":"object","properties":{"city":{"type":"string"}},"required":["city"]}}}]
base = {"id": "r1", "metadata": {"source": "test", "task_type": "tool_calling"}, "tools": tools, "messages": [
 {"role":"user","content":"Weather in Paris?"},
 {"role":"assistant","reasoning_content":"Call the tool.","content":"","tool_calls":[{"id":"c1","type":"function","function":{"name":"get_weather","arguments":{"city":"Paris"}}}]},
 {"role":"tool","tool_call_id":"c1","name":"get_weather","content":"18C, sunny"},
 {"role":"assistant","reasoning_content":"Got it.","content":"It's 18C and sunny in Paris."}]}

def show(name, rec, **cfg):
    r = ChatTokenizer(tok, TokenizerConfig(**cfg))(rec)
    print(f"{name:22} mode={r.mode!s:9} samples={len(r.samples)} fallback={r.fallback_reason!r} "
          f"rejections={[(x.reason.value, x.turn_idx) for x in r.rejections]}")

no_r = copy.deepcopy(base); [m.pop("reasoning_content", None) for m in no_r["messages"]]
frozen = copy.deepcopy(base); [m.__setitem__("train", False) for m in frozen["messages"]]
half = copy.deepcopy(base); half["messages"][1]["train"] = False

show("reasoning (auto)", base)                       # single, 1 sample
show("no_reasoning (auto)", no_r)                    # per_turn, 2 samples, fallback set
show("no_reasoning (single)", no_r, mode="single")   # rejected prefix_mismatch
show("reasoning (per_turn)", base, mode="per_turn")  # per_turn, 2 samples, fallback None
show("overlong 200 (auto)", base, max_seq_len=200)   # per_turn, 1 sample + 1 overlong rejection
show("all frozen", frozen)                           # rejected no_train_turns
show("turn 1 frozen", half)                          # single, 1 sample
show("no metadata", {"id": "x", "messages": base["messages"]})   # works
show("missing messages", {"id": "bad1"})                         # rejected, must NOT crash
show("missing role", {"id": "bad2", "messages": [{"content": "hi"}]})  # rejected, must NOT crash
before = copy.deepcopy(base); ChatTokenizer(tok)(base); print("input unchanged:", before == base)
try:
    ChatTokenizer(tok, TokenizerConfig(raise_errors=True))({"id": "bad1"}); print("raise_errors: did NOT raise ❌")
except KeyError as e:
    print("raise_errors: raised KeyError ✅", e)

/opt/conda/envs/vllm_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


reasoning (auto)       mode=single    samples=1 fallback=None rejections=[]
no_reasoning (auto)    mode=None      samples=0 fallback=None rejections=[('tokenize_error', None)]
no_reasoning (single)  mode=None      samples=0 fallback=None rejections=[('tokenize_error', None)]
reasoning (per_turn)   mode=per_turn  samples=2 fallback=None rejections=[]
overlong 200 (auto)    mode=None      samples=0 fallback=None rejections=[('tokenize_error', None)]
all frozen             mode=None      samples=0 fallback=None rejections=[('no_train_turns', None)]
turn 1 frozen          mode=single    samples=1 fallback=None rejections=[]
no metadata            mode=single    samples=1 fallback=None rejections=[]


TypeError: 'NoneType' object is not iterable